In [1]:
# This file is a jupytext-paired Python script export of
# `using_model_outputs_solution.ipynb`. The canonical artifact for learners is
# the notebook (.ipynb); this script is provided for code review and `git diff`
# readability. Run `jupytext --sync` to keep the two in lockstep after edits.

# Using Model Outputs in Decision Calculations (SOLUTION)

## Scenario

You are a decision scientist at **ClearBridge Lending** (fictional), an online
personal lending marketplace. ClearBridge's credit-risk team has built a
gradient boosting classifier trained on three years of historical loan
outcomes. For each new application, the model outputs a **probability of
default (PD)** — the estimated probability that the borrower will not repay.

The file `../using-model-outputs-starter/data/loan_applicants.csv` contains
model output for 500 pending applications: `applicant_id`, `predicted_pd`,
`loan_amount_usd`, and the `annual_interest_rate` ClearBridge would offer
each applicant.

The business question: **what probability-of-default threshold should
ClearBridge use?** Applications below the threshold get approved; above it
get rejected.

Three candidate thresholds:

- **Strict** — approve if PD < 15%
- **Moderate** — approve if PD < 25%
- **Permissive** — approve if PD < 40%

The deliverable is a quantified comparison of all three thresholds — approval
rate and expected total profit — and a defended recommendation.

## Setup

In [2]:
import numpy as np
import pandas as pd

DATA_PATH = "../using-model-outputs-starter/data/loan_applicants.csv"

# Constants — loan product parameters
LOAN_TERM_YEARS = 3        # 3-year personal loan
LOSS_GIVEN_DEFAULT = 0.75  # 75% of principal lost on default; 25% recovered

# Approval thresholds
THRESHOLDS = {
    "Strict":     0.15,
    "Moderate":   0.25,
    "Permissive": 0.40,
}

## 1. Load the data

In [3]:
applicants = pd.read_csv(DATA_PATH)
display(applicants.head())
print(f"Shape: {applicants.shape}")

,applicant_id,predicted_pd,loan_amount_usd,annual_interest_rate
0,CB-001,0.2148,5021,0.0883
1,CB-002,0.2461,24728,0.1035
2,CB-003,0.3458,7181,0.1290
3,CB-004,0.2625,3357,0.0834
4,CB-005,0.3701,4712,0.1370


Shape: (500, 4)


## 2. Implement `applicant_ev()`

The expected net profit from approving a single applicant is:

$$\text{EV} = \text{loan\_amount} \times \text{annual\_rate} \times
\text{LOAN\_TERM\_YEARS} \times (1 - \text{PD}) -
\text{loan\_amount} \times \text{LGD} \times \text{PD}$$

The first term is expected interest income — earned in full if the borrower
repays, zero otherwise. The second term is expected principal loss — incurred
in full (less recoveries) if the borrower defaults, zero otherwise. LGD = 0.75
means 75% of principal is lost; 25% is recovered via collections.

In [4]:
def applicant_ev(predicted_pd: float, loan_amount: float, annual_rate: float,
                 loan_term_years: float = LOAN_TERM_YEARS,
                 lgd: float = LOSS_GIVEN_DEFAULT) -> float:
    """Expected net profit from approving one loan applicant.

    Parameters
    ----------
    predicted_pd : float
        Model-output probability of default (0–1).
    loan_amount : float
        Requested principal in USD.
    annual_rate : float
        Annual interest rate offered, in decimal form (e.g., 0.12 = 12%).
    loan_term_years : float
        Loan term in years.
    lgd : float
        Loss given default as a fraction of principal (e.g., 0.75 means
        75% of principal is lost; 25% is recovered via collections).

    Returns
    -------
    float
        Expected net profit in USD.
    """
    interest_income = loan_amount * annual_rate * loan_term_years * (1 - predicted_pd)
    expected_loss = loan_amount * lgd * predicted_pd
    return interest_income - expected_loss

## 3. Apply `applicant_ev()` to the full dataset

In [5]:
applicants["ev_usd"] = applicant_ev(
    applicants["predicted_pd"],
    applicants["loan_amount_usd"],
    applicants["annual_interest_rate"],
)
display(applicants.head())

,applicant_id,predicted_pd,loan_amount_usd,annual_interest_rate,ev_usd
0,CB-001,0.2148,5021,0.0883,235.482289
1,CB-002,0.2461,24728,0.1035,1224.306772
2,CB-003,0.3458,7181,0.1290,-44.339803
3,CB-004,0.2625,3357,0.0834,-41.467343
4,CB-005,0.3701,4712,0.1370,-88.048903


## 4. Compare the three thresholds

In [6]:
rows = []
for name, thresh in THRESHOLDS.items():
    approved = applicants[applicants["predicted_pd"] < thresh]
    rows.append({
        "threshold": name,
        "pd_cutoff": thresh,
        "n_approved": len(approved),
        "approval_rate_pct": round(100 * len(approved) / len(applicants), 1),
        "total_ev_usd": approved["ev_usd"].sum(),
    })

comparison = pd.DataFrame(rows).set_index("threshold")

## 5. Display the comparison table

In [7]:
display(
    comparison.style.format({
        "pd_cutoff": "{:.0%}",
        "approval_rate_pct": "{:.1f}%",
        "total_ev_usd": "${:,.0f}",
    })
)

,pd_cutoff,n_approved,approval_rate_pct,total_ev_usd
threshold,,,,
Strict,15%,137,27.4%,"$260,341"
Moderate,25%,272,54.4%,"$416,960"
Permissive,40%,416,83.2%,"$416,626"


## 6. Identify the profit-maximizing threshold

In [8]:
best = comparison["total_ev_usd"].idxmax()
best_ev = comparison.loc[best, "total_ev_usd"]
print(f"Profit-maximizing threshold: {best}  (total EV = ${best_ev:,.0f})")

Profit-maximizing threshold: Moderate  (total EV = $416,960)


**Moderate (PD < 25%) maximizes expected total profit.** Strict leaves money
on the table: the 135 additional applicants approved under Moderate have
positive mean EV, so including them adds ~$157K to portfolio profit.
Permissive adds a further 144 applicants in the 25%–40% PD band whose mean
expected EV is essentially zero — the higher default losses on that group
just offset their interest income, leaving total profit flat or slightly
below Moderate.

## 7. Sensitivity flex — higher loss severity

The 75% LGD baseline assumes typical collections performance. In a soft
economic environment, recovery rates fall and LGD could reach 85%.

In [9]:
LGD_HIGH = 0.85

rows_flex = []
for name, thresh in THRESHOLDS.items():
    approved = applicants[applicants["predicted_pd"] < thresh].copy()
    approved["ev_high_lgd"] = applicant_ev(
        approved["predicted_pd"],
        approved["loan_amount_usd"],
        approved["annual_interest_rate"],
        lgd=LGD_HIGH,
    )
    rows_flex.append({
        "threshold": name,
        "total_ev_base_usd": applicants[applicants["predicted_pd"] < thresh]["ev_usd"].sum(),
        "total_ev_high_lgd_usd": approved["ev_high_lgd"].sum(),
    })

flex = pd.DataFrame(rows_flex).set_index("threshold")
display(
    flex.style.format("${:,.0f}")
)

best_flex = flex["total_ev_high_lgd_usd"].idxmax()
print(f"\nProfit-maximizing threshold at LGD = {LGD_HIGH:.0%}: {best_flex}")

,total_ev_base_usd,total_ev_high_lgd_usd
threshold,,
Strict,"$260,341","$243,876"
Moderate,"$416,960","$360,536"
Permissive,"$416,626","$301,838"



Profit-maximizing threshold at LGD = 85%: Moderate


**Moderate remains the profit-maximizing threshold even when LGD rises to
85%.** Under tougher collection conditions, the 25%–40% PD band — already
marginal at 75% LGD — turns clearly negative, widening the gap between
Moderate and Permissive. The recommendation becomes more robust, not less,
as loss severity increases.

## Nimbus Streaming preview

This analysis is a direct input to the cost-benefit model in a full decision
pipeline. In **Nimbus Streaming**, the same `applicant_ev()` function runs
over live application data as it arrives, updating the threshold recommendation
in real time as portfolio composition shifts. The comparison table above becomes
a live dashboard refreshed each time the credit-risk model scores a new batch
of applications.